# CountVision – GPU benchmark and RF-DETR accuracy (Google Colab, T4)

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then run the cells from top to bottom.

What this notebook does (about 15–25 minutes, costs €0):
1. Installs CountVision from GitHub (your repo `SamiHotak/countvision`).
2. Downloads the 6 evaluation clips (Intel sample videos, CC BY 4.0) and the YOLOX models (Apache-2.0).
3. **Speed:** detector × runtime × input size on the T4 → `eval/bench/colab-t4.json`.
4. **Accuracy:** RF-DETR Nano/Small (Apache-2.0) and YOLOX-S on the same labeled clips → `eval/runs/*.json`.
5. Zips the results. Download the zip and copy its files into your repo (see the last cell).

Known Colab issues (from earlier projects): restart the session after the first `pip install` cell
if Colab asks; pip warnings about pre-installed packages are harmless; the `libnvrtc-builtins.so.13`
error is fixed by the preload cell.

In [ ]:
# 1) Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Get the code and install it (CPU/GPU detectors: ultralytics, onnxruntime-gpu, rfdetr)
%cd /content
!rm -rf countvision && git clone -q https://github.com/SamiHotak/countvision.git
%cd /content/countvision
!pip -q install -e "edge[yolo]" rfdetr
!pip -q uninstall -y onnxruntime  # the CPU build would hide the GPU build
!pip -q install onnxruntime-gpu
# If Colab shows a "Restart session" button now: click it, then continue with the next cell.

In [ ]:
# 3) Work-around for CUDA 13 on Colab: "failed to open libnvrtc-builtins.so.13.0"
import ctypes, glob, os, sys
loaded = []
for folder in sys.path:
    for path in glob.glob(os.path.join(folder, "nvidia", "*", "lib", "libnvrtc-builtins.so.*")):
        try:
            ctypes.CDLL(path, mode=ctypes.RTLD_GLOBAL); loaded.append(os.path.basename(path))
        except OSError:
            pass
print("preloaded:", loaded or "nothing needed")
import torch, onnxruntime as ort
print("torch", torch.__version__, "CUDA", torch.version.cuda, "GPU:", torch.cuda.is_available())
print("onnxruntime providers:", ort.get_available_providers())

In [ ]:
# 4) Videos (CC BY 4.0, Intel IoT DevKit sample-videos) and models
%cd /content/countvision
!countvision-edge eval fetch
!mkdir -p eval/models
!for m in yolox_nano yolox_tiny yolox_s; do wget -q -nc -O eval/models/$m.onnx https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/$m.onnx; done
!ls -la eval/videos eval/models

In [ ]:
# 5) SPEED on the T4: detector x runtime x input size (60 real frames, 1280x720)
%cd /content/countvision
!countvision-edge bench --label colab-t4 --device cuda --video eval/videos/store-aisle-detection.mp4 \
  --model ultralytics:yolo11n.pt@640 --model ultralytics:yolo11n.pt@416 \
  --model ultralytics:yolo11s.pt@640 \
  --model onnx:yolox_tiny.onnx --model onnx:yolox_s.onnx \
  --model rfdetr:nano --model rfdetr:small --model rfdetr:medium

`cams@15` in the table = how many cameras one T4 could run at 15 FPS **with one model call per frame,
one after the other** (no batching). Target from the plan: ≥ 4 cameras at 15 FPS.

In [ ]:
# 6) ACCURACY of the permissive models on the same labeled clips
#    Rule: tune on the TUNE clips only, then score the TEST clips once.
%cd /content/countvision
for spec in ["rfdetr:nano", "rfdetr:small", "onnx:yolox_s.onnx"]:
    key = spec.replace(":", "_")
    !countvision-edge eval tune --detector {spec} --device cuda
    for split in ["tune", "test"]:
        !countvision-edge eval run --detector {spec} --device cuda --split {split} --tag after --params eval/params/{key}.yaml --note "Colab T4"

In [ ]:
# 7) Pack the results and download them
%cd /content/countvision
!zip -qr /content/countvision_colab_results.zip eval/bench eval/runs eval/params
from google.colab import files
files.download("/content/countvision_colab_results.zip")

## Then, on your laptop

1. Unzip `countvision_colab_results.zip` **into your repo folder** (it contains `eval/bench`, `eval/runs`, `eval/params`).
2. Run `countvision-edge eval report` — this rewrites the tables in `eval/results.md`.
3. Commit and push.